# FLAG 2027 - Notebook 19: MFV-KSD (FAME 2024 winner) as a fusion member

Dr. Ruijie Tao gave written permission (25 Oct) to use the released `seen.pt` model for inference. It is a cross-modally fine-tuned ECAPA + IResNet-18 + fusion head (ACM MM 2024, arXiv 2407.17902), trained on VoxCeleb1/2 + MAV-Celeb (En/Ur/Hi). No Bengali training -> rule-compliant; declared in the system description.

1. Clone the repo, load `seen.pt` into their `trainer` class.
2. Score pairs exactly as their `test_network` does (full clip + 2-s windows averaged; face 112x112 BGR in [-1,1]).
3. **Sign check on CHALLENGE TRAINING pairs** (labels known): pick the score direction that gives EER < 50.
4. Dev scores for the 4 cells -> ZIP for MFV-KSD alone, and fixed-weight fusions with our P1 system (weights from TRAINING-pair statistics only; no test statistics - FAQ-compliant).

Before running: add a Drive shortcut to Dr. Tao's folder https://drive.google.com/drive/folders/1FFyHesc9li-UY7qsFKsCd3fEZ5nTI2rx (Organize -> Add shortcut -> My Drive). About 30 min. Paste the **SUMMARY**.

In [ ]:
# Cell 1 - Drive, GPU, clone the repo
from google.colab import drive
drive.mount('/content/drive')
import os, sys, subprocess, torch
if not os.path.exists('/content/MFV-KSD'):
    subprocess.run(['git', 'clone', '-q', 'https://github.com/TaoRuijie/MFV-KSD.git', '/content/MFV-KSD'], check=True)
os.chdir('/content/MFV-KSD'); sys.path.insert(0, '/content/MFV-KSD')
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', dev, '| repo files:', sorted(os.listdir('/content/MFV-KSD')))

In [ ]:
# Cell 2 - find the pretrained weights in your Drive (shortcut to Dr. Tao's folder) and copy them locally
import glob, shutil
BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027'); RESDIR = os.path.join(PROJECT, 'results')
SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)
cands = []
for top in os.listdir(BASE):
    p = os.path.join(BASE, top)
    if not os.path.isdir(p) or top in ('FLAG2027', 'Videos', 'FairPersonalityThesis', 'VLM_Personality', 'Colab Notebooks', 'dataset'):
        continue
    for r, d, fs in os.walk(p, followlinks=True):
        if r[len(p):].count(os.sep) > 2:
            d[:] = []; continue
        for f in fs:
            if f.endswith(('.pt', '.model', '.pth')):
                cands.append(os.path.join(r, f))
print('model files found in Drive shortcuts:')
for c in cands:
    print('   %-80s %.0f MB' % (c[len(BASE):], os.path.getsize(c) / 1e6))
seen = [c for c in cands if 'seen' in os.path.basename(c).lower() and 'unseen' not in os.path.basename(c).lower()]
if not seen:
    raise SystemExit('No file with "seen" in its name found. Add the Drive shortcut to the MFV-KSD folder, then re-run. If the files have other names, paste the list above to Claude.')
os.makedirs('/content/MFV-KSD/pretrain', exist_ok=True)
WEIGHTS = '/content/MFV-KSD/pretrain/seen.pt'; shutil.copy(seen[0], WEIGHTS)
note('Using weights: ' + seen[0][len(BASE):])

In [ ]:
# Cell 3 - build their trainer and load seen.pt (their loader is partial: it reports keys it could not match)
import types, numpy as np, cv2, soundfile as sf, librosa, io, contextlib
from trainer import trainer as MFVTrainer
args = types.SimpleNamespace(lr=1e-5, lr_decay=0.97, test_step=1, pretrain_s='', pretrain_f='', initial_model=WEIGHTS)
buf = io.StringIO()
with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
    M = MFVTrainer(args)
    sd = torch.load(WEIGHTS, map_location=dev)
    own = M.state_dict(); n_ok = 0; missing = []
    for k, v in sd.items():
        kk = k if k in own else k.replace('module.', '')
        if kk in own and own[kk].shape == v.shape:
            own[kk].copy_(v); n_ok += 1
        else:
            missing.append(k)
M.eval()
note('seen.pt: %d of %d tensors loaded into the model (%d unmatched)' % (n_ok, len(sd), len(missing)))
if missing[:5]:
    print('unmatched examples:', missing[:5])
print(buf.getvalue()[-600:])

In [ ]:
# Cell 4 - scoring exactly like their test_network (full clip + 2-s windows at 1-s hop, averaged)
@torch.no_grad()
def face_tensor(path, mode='test'):
    img = cv2.resize(cv2.imread(path), (112, 112))                 # BGR, as in their code
    t = torch.FloatTensor(np.transpose(img, (2, 0, 1))).unsqueeze(0)
    if mode == 'test':
        t = t.div(255).sub(0.5).div(0.5)                           # their test-time normalisation
    return t.to(dev)
@torch.no_grad()
def score_pair(voice_path, face_path, mode='test'):
    speech, sr = sf.read(voice_path)
    if speech.ndim > 1:
        speech = speech.mean(1)
    if sr != 16000:
        speech = librosa.resample(speech.astype(np.float32), orig_sr=sr, target_sr=16000)
    speech = speech.astype(np.float32)
    f_emb = M.face_encoder.forward(face_tensor(face_path, mode))
    def one(seg):
        s_emb = M.speaker_encoder.forward(torch.FloatTensor(seg).unsqueeze(0).to(dev))
        av = M.fusion_module.forward(s_emb, f_emb)
        return float(M.loss.forward(av)[0])
    s1 = one(speech)
    max_audio = 3 * 16000
    if speech.shape[0] < max_audio:
        speech = np.pad(speech, (0, max_audio - speech.shape[0]), 'wrap')
    wins = [speech[a * 16000:(a + 2) * 16000] for a in range(0, int(speech.shape[0] // 16000) - 1)]
    s2 = np.mean([one(w) for w in wins]) if wins else s1
    return (s1 + s2) / 2
# smoke test on one pair
import json, zipfile
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
FULL = '/content/flag_data'; os.makedirs(FULL, exist_ok=True)
for z in ['train_set.zip', 'dev_set.zip']:
    if not os.path.exists(os.path.join(FULL, z + '.done')):
        with zipfile.ZipFile(os.path.join(ZIP_DIR, z)) as zf:
            zf.extractall(FULL)
        open(os.path.join(FULL, z + '.done'), 'w').close()
TXT = [os.path.join(r, f) for r, _, fs in os.walk(FULL) for f in fs if f.endswith('.txt')]
def find_one(*parts):
    hits = [p for p in TXT if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]
def resolve(rel, list_path):
    q = os.path.dirname(list_path)
    while q.startswith(FULL):
        c = os.path.join(q, rel)
        if os.path.exists(c):
            return c
        q = os.path.dirname(q)
tl = find_one('train_English.txt'); train_lines = [l.split() for l in open(tl) if l.strip()]
print('smoke test score:', score_pair(resolve(train_lines[0][2], tl), resolve(train_lines[0][3], tl)))

In [ ]:
# Cell 5 - SIGN CHECK on challenge TRAINING pairs (labels known): which direction means 'same person'?
# also records the score SPREAD on training pairs (used later as a fixed fusion weight; no test statistics)
from sklearn.metrics import roc_curve
from tqdm.auto import tqdm
def eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores); fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr)); return (fpr[i] + fnr[i]) / 2 * 100
spk = np.array([t[4] for t in train_lines]); people = np.unique(spk); rows = {s: np.where(spk == s)[0] for s in people}
rng = np.random.default_rng(0); pairs = []
for _ in range(400):
    s = rng.choice(people); a, b = rng.choice(rows[s], 2, replace=False); pairs.append((a, b, 1))
    s, t = rng.choice(people, 2, replace=False); pairs.append((rng.choice(rows[s]), rng.choice(rows[t]), 0))
SIGN, TRAIN_STD = {}, {}
for mode in ['test', 'raw255']:
    sc = np.array([score_pair(resolve(train_lines[a][2], tl), resolve(train_lines[b][3], tl), mode) for a, b, _ in tqdm(pairs, desc='sign check ' + mode)])
    lab = np.array([l for _, _, l in pairs])
    e_plus, e_minus = eer(lab, sc), eer(lab, -sc)
    SIGN[mode] = 1.0 if e_plus < e_minus else -1.0
    TRAIN_STD[mode] = float(np.std(sc))
    note('train pairs (800), face mode %-6s: EER %.1f with +score, %.1f with -score -> sign %+d | train score std %.3f' % (mode, e_plus, e_minus, int(SIGN[mode]), TRAIN_STD[mode]))

In [ ]:
# Cell 6 - dev scores (4 cells), ZIP for MFV-KSD alone (both face modes), fixed-weight fusions with P1
SPLIT_NAMES = ['no_gender_English', 'no_gender_Bangla', 'gender_English', 'gender_Bangla']
LANG_FILE = {'English': 'sub_score_v4_English_heard.txt', 'Bangla': 'sub_score_v4_Bangla_unheard.txt'}
SUBDIR = os.path.join(PROJECT, 'submissions'); os.makedirs(SUBDIR, exist_ok=True)
def write_zip(name, scores):   # scores: HIGHER = same; files get LOWER = same
    OUT = '/content/sub_' + name
    for track in ['no_gender', 'gender']:
        os.makedirs(os.path.join(OUT, track), exist_ok=True)
        for lang_, fname in LANG_FILE.items():
            split = track + '_' + lang_
            pair_ids = [l.split()[0] for l in open(find_one('/' + track + '/', lang_ + '_test.txt')) if l.strip()]
            assert len(pair_ids) == len(scores[split]) and np.isfinite(scores[split]).all()
            with open(os.path.join(OUT, track, fname), 'w') as f:
                for pid, v in zip(pair_ids, -scores[split]):
                    f.write('%s %.6f\n' % (pid, v))
    zp = os.path.join(SUBDIR, name + '.zip')
    with zipfile.ZipFile(zp, 'w', zipfile.ZIP_DEFLATED) as zf:
        for track in ['no_gender', 'gender']:
            for fname in LANG_FILE.values():
                zf.write(os.path.join(OUT, track, fname), arcname=track + '/' + fname)
    note('Submission ZIP saved: ' + zp[len(BASE):])
MF = {}
for mode in ['test', 'raw255']:
    sc = {}
    for split in SPLIT_NAMES:
        track, lang_ = split.rsplit('_', 1); lp = find_one('/' + track + '/', lang_ + '_test.txt')
        items = [(resolve(l.split()[1], lp), resolve(l.split()[2], lp)) for l in open(lp) if l.strip()]
        sc[split] = SIGN[mode] * np.array([score_pair(v, f, mode) for v, f in tqdm(items, desc='%s %s' % (mode, split))])
    MF[mode] = sc
    np.savez(os.path.join(RESDIR, 'devscores_19_mfvksd_%s.npz' % mode), **sc)
    write_zip('19_MFVKSD_%s' % mode, sc)
# Fusion with P1. The weight is FIXED from TRAINING-pair spreads (Cell 5), never from dev/eval scores (FAQ rule).
# Our P1 score is -L2 between tanh embeddings; its spread on training pairs is ~0.55 (measured in Notebook 15's validation).
p15 = os.path.join(RESDIR, 'devscores_15_P1_proto.npz')
if os.path.exists(p15):
    P1 = dict(np.load(p15)); OUR_STD = 0.55
    for mode in ['test', 'raw255']:
        w = OUR_STD / TRAIN_STD[mode]
        for name, wt in [('equal', w), ('half', 0.5 * w)]:
            write_zip('19_ENS_P1+MFVKSD_%s_%s' % (mode, name), {s: P1[s] + wt * MF[mode][s] for s in SPLIT_NAMES})
        note('fixed fusion weight (%s mode): %.3f equal-spread, %.3f half  [from training-pair std only]' % (mode, w, 0.5 * w))
print('=' * 70)
print('SUMMARY  (copy everything between the lines and paste to Claude)')
print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)